In [1]:
import numpy as np

In [2]:
#estimate gaussian paramenters
def estimate_gaussian(x):
    m, n = x.shape
    mu = np.mean(x, axis = 0)
    var = np.mean((x-mu)**2, axis = 0)
    return mu, var

In [3]:
#multivariate gaussian probability
def multivariate_gaussian(x, mu, var):
    n = len(mu)
    sigma = np.diag(var)
    x = x - mu
    coeff = 1/((2*np.pi)**(n/2)*np.linalg.det(sigma)**0.5)
    exp_term = np.exp(-0.5*np.sum(x @ np.linalg.inv(sigma)*x, axis = 1))
    return coeff*exp_term

In [4]:
#select threshold epsilon using F1-score
def select_threshold(y_val, p_val):
    best_epsilon = 0
    best_F1 = 0
    step = (max(p_val) - min(p_val)) / 1000
    for eps in np.arange(min(p_val), max(p_val), step):
        preds = (p_val < eps)
        tp = np.sum((preds == 1) & (y_val == 1))
        fp = np.sum((preds == 1) & (y_val == 0))
        fn = np.sum((preds == 0) & (y_val == 1))
        
        if tp + fp == 0 or tp + fn == 0:
            continue
            
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        F1 = 2*prec*rec / (prec + rec)

        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = eps
            
    return best_epsilon, best_F1

In [6]:
# dataset
X_train = np.array([
    [13.0, 14.5],
    [14.2, 15.1],
    [13.8, 14.9],
    [14.0, 15.0],
    [13.9, 14.8],
    [14.1, 15.2],
    [13.7, 14.6],
    [14.3, 15.1]
])

X_val = np.array([
    [14.0, 15.0],   # normal
    [13.8, 14.7],   # normal
    [14.2, 15.1],   # normal
    [20.0, 5.0],    # anomaly
    [2.0, 25.0]     # anomaly
])

y_val = np.array([
    0,  # normal
    0,  # normal
    0,  # normal
    1,  # anomaly
    1   # anomaly
])

In [7]:
# full workflow
# X_train: normal data
# X_val, y_val: validation data (y=1 anomaly, y=0 normal)

mu, var = estimate_gaussian(X_train)

p_train = multivariate_gaussian(X_train, mu, var)
p_val = multivariate_gaussian(X_val, mu, var)

epsilon, F1 = select_threshold(y_val, p_val)

outliers = p_train < epsilon
print("Best epsilon:", epsilon)
print("F1 score:", F1)
print("Number of anomalies detected:", np.sum(outliers))

Best epsilon: 0.001544936693295418
F1 score: 1.0
Number of anomalies detected: 0
